# Semana 12: Regresión - Predecir el precio

Usamos aprendizaje supervisado (regresión lineal) para predecir el precio de un alojamiento a partir de sus características. La idea de negocio: dado un alojamiento con cierta puntuación y tipo, ¿cuánto debería costar? Esto sirve para detectar alojamientos sobrevalorados o subvalorados respecto a lo que el modelo espera.

Predecimos `precio_clp` a partir de la puntuación y el tipo de alojamiento.

## Conexión y carga de datos

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

spark = SparkSession.builder \
    .appName("Regresion_Alojamientos") \
    .config("spark.mongodb.read.connection.uri", MONGO_URI) \
    .config("spark.mongodb.write.connection.uri", MONGO_URI) \
    .getOrCreate()

df = spark.read.format("mongodb") \
    .option("database", "proyecto_bigdata") \
    .option("collection", "processed_data") \
    .load()

print("Registros cargados:", df.count())
df.select("precio_clp", "puntuacion", "tipo_alojamiento").show(10)

Registros cargados: 3584
+----------+----------+----------------+
|precio_clp|puntuacion|tipo_alojamiento|
+----------+----------+----------------+
|   82822.0|       7.0|     apartamento|
|   79329.0|       7.1|     apartamento|
|  158700.0|       7.1|     apartamento|
|   80781.0|       9.5|     apartamento|
|  118247.0|       7.3|     apartamento|
|   63625.0|       9.6|     apartamento|
|   97591.0|       7.5|     apartamento|
|   78879.0|       7.3|     apartamento|
|  106047.0|       7.4|     apartamento|
|   96889.0|       8.7|     apartamento|
+----------+----------+----------------+
only showing top 10 rows



## Preparar los datos

El tipo de alojamiento es texto, así que lo convertimos a número con StringIndexer para que la regresión lo pueda usar. La puntuación ya es número. Juntamos ambas en un vector de características y las escalamos.

In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

df_reg = df.dropna(subset=["precio_clp", "puntuacion", "tipo_alojamiento"])

indexer = StringIndexer(inputCol="tipo_alojamiento", outputCol="tipo_idx")
df_reg = indexer.fit(df_reg).transform(df_reg)

assembler = VectorAssembler(inputCols=["puntuacion", "tipo_idx"], outputCol="features")
df_vector = assembler.transform(df_reg)

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
df_final = scaler.fit(df_vector).transform(df_vector)

df_final = df_final.withColumnRenamed("precio_clp", "label_precio")
print("Datos preparados")

Datos preparados


## Dividir en entrenamiento y prueba

Separamos los datos: 70% para que el modelo aprenda y 30% para probarlo con datos que no vio.

In [3]:
train, test = df_final.randomSplit([0.7, 0.3], seed=42)
print("Entrenamiento:", train.count())
print("Prueba:", test.count())

Entrenamiento: 2568
Prueba: 1016


## Entrenar la regresión lineal

In [4]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="scaledFeatures", labelCol="label_precio", maxIter=10)
lr_model = lr.fit(train)

predicciones = lr_model.transform(test)
print("=== PRECIO REAL VS PRECIO PREDICHO ===")
predicciones.select("tipo_alojamiento", "puntuacion", "label_precio", "prediction").show(10)

=== PRECIO REAL VS PRECIO PREDICHO ===
+----------------+----------+------------+-----------------+
|tipo_alojamiento|puntuacion|label_precio|       prediction|
+----------------+----------+------------+-----------------+
|     apartamento|       7.1|    158700.0|71276.25625339337|
|     apartamento|       7.5|     97591.0|73277.44602263535|
|     apartamento|       7.4|    106047.0|72777.14858032487|
|     apartamento|       8.7|     96889.0| 79281.0153303613|
|     apartamento|       9.2|    124407.0|81782.50254191378|
|     apartamento|       9.6|     67434.0|83783.69231115577|
|     apartamento|       7.8|    107164.0|74778.33834956684|
|     apartamento|       7.7|     51220.0|74278.04090725633|
|     apartamento|       7.8|    214500.0|74778.33834956684|
|     apartamento|       7.5|     50879.0|73277.44602263535|
+----------------+----------+------------+-----------------+
only showing top 10 rows



## Evaluar el modelo

Medimos qué tan bien predice con dos métricas: R² (qué porcentaje del precio explica el modelo, más alto mejor) y RMSE (el error promedio en pesos, más bajo mejor).

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

ev_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
ev_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = ev_r2.evaluate(predicciones)
rmse = ev_rmse.evaluate(predicciones)

print("="*50)
print("     EVALUACIÓN DE LA REGRESIÓN")
print("="*50)
print(f"R² (qué tanto explica el modelo): {r2*100:.2f}%")
print(f"RMSE (error promedio en pesos):   {rmse:,.0f}")
print("="*50)

     EVALUACIÓN DE LA REGRESIÓN
R² (qué tanto explica el modelo): 6.38%
RMSE (error promedio en pesos):   50,512


## Coeficientes: ¿cuánto pesa cada característica?

El modelo le asigna un peso a cada variable. Esto nos dice cuánto influye la puntuación y el tipo en el precio.

In [7]:
print(f"Precio base (intersección): {lr_model.intercept:,.0f}")
print(f"Peso de la puntuación:      {lr_model.coefficients[0]:,.2f}")
print(f"Peso del tipo:              {lr_model.coefficients[1]:,.2f}")

Precio base (intersección): 56,508
Peso de la puntuación:      4,059.53
Peso del tipo:              -10,617.82


## Comparar con otros modelos

La regresión lineal asume que la relación es una línea recta. Probamos también un Árbol de Decisión y un Random Forest, que captan relaciones más complejas, para ver si alguno predice mejor el precio. Comparamos los tres con las mismas métricas (R² y RMSE).

In [8]:
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor

# Arbol de decision
dt = DecisionTreeRegressor(featuresCol="scaledFeatures", labelCol="label_precio")
dt_model = dt.fit(train)
pred_dt = dt_model.transform(test)

# Random Forest
rf = RandomForestRegressor(featuresCol="scaledFeatures", labelCol="label_precio", seed=42)
rf_model = rf.fit(train)
pred_rf = rf_model.transform(test)

print("Modelos entrenados: Árbol de Decisión y Random Forest")

Modelos entrenados: Árbol de Decisión y Random Forest


In [9]:
# Evaluar los tres modelos con las mismas metricas
modelos = {
    "Regresión Lineal": predicciones,
    "Árbol de Decisión": pred_dt,
    "Random Forest": pred_rf
}

print("="*55)
print(f"{'Modelo':<20}{'R²':>12}{'RMSE':>18}")
print("="*55)
for nombre, pred in modelos.items():
    r2_m = ev_r2.evaluate(pred)
    rmse_m = ev_rmse.evaluate(pred)
    print(f"{nombre:<20}{r2_m*100:>10.2f}%{rmse_m:>17,.0f}")
print("="*55)

Modelo                        R²              RMSE
Regresión Lineal          6.38%           50,512
Árbol de Decisión        23.43%           45,681
Random Forest            23.93%           45,533


## Conclusiones

(Completar según los resultados: qué tan bien predijo el modelo, qué significa el R² obtenido, y qué nos dice sobre la relación entre las características y el precio. Si el R² sale bajo, significa que la puntuación y el tipo no alcanzan para explicar el precio, lo cual es un hallazgo válido: el precio de un alojamiento depende de más factores que no están en los datos, como la ubicación exacta, la temporada o los servicios.)